In [17]:
import pandas as pd

import sqlite3

conn = sqlite3.connect('../data/olist.db')

query = """
SELECT name FROM sqlite_master WHERE type = 'table'
"""

df_resultado = pd.read_sql(query, conn)
df_resultado

,name
0,customers
1,orders
2,order_items
3,order_payments
4,order_reviews
5,products
6,sellers
7,geolocation
8,category_translation


In [89]:
query = """
SELECT ROUND(SUM(oi.price)) AS total, COUNT(DISTINCT o.order_id) AS numeroPedidos, customer_state AS estadoCliente
FROM order_items AS oi
JOIN orders AS o
ON oi.order_id = o.order_id
JOIN customers AS c
ON o.customer_id = c.customer_id
WHERE order_status = 'delivered'
GROUP BY customer_state
ORDER BY total DESC
LIMIT 10
"""


df_resultado = pd.read_sql(query, conn)
df_resultado

,total,numeroPedidos,estadoCliente
0,5067633.0,40501,SP
1,1759651.0,12350,RJ
2,1552482.0,11354,MG
3,728897.0,5345,RS
4,666064.0,4923,PR
5,507012.0,3546,SC
6,493584.0,3256,BA
7,296498.0,2080,DF
8,282837.0,1957,GO
9,268643.0,1995,ES


In [39]:
query = """

SELECT ct.product_category_name, SUM(oi.price) AS faturamentoCateg
FROM category_translation AS ct 
JOIN products AS p
ON ct.product_category_name = p.product_category_name
JOIN order_items AS oi
ON oi.product_id = p.product_id
GROUP BY ct.product_category_name
ORDER BY faturamentoCateg DESC 
LIMIT 10
"""

df_resultado = pd.read_sql(query, conn)
df_resultado


,product_category_name,faturamentoCateg
0,beleza_saude,1258681.34
1,relogios_presentes,1205005.68
2,cama_mesa_banho,1036988.68
3,esporte_lazer,988048.97
4,informatica_acessorios,911954.32
5,moveis_decoracao,729762.49
6,cool_stuff,635290.85
7,utilidades_domesticas,632248.66
8,automotivo,592720.11
9,ferramentas_jardim,485256.46


In [62]:
query = """

WITH contagem AS (
SELECT COUNT(CASE WHEN o.order_status = 'canceled' THEN 1 END ) AS cancelados,
COUNT(o.order_id) AS total, p.product_category_name
FROM orders AS o

JOIN order_items AS oi
ON o.order_id = oi.order_id

JOIN products AS p
ON oi.product_id = p.product_id

GROUP BY p.product_category_name
)

SELECT cancelados, total, product_category_name,
ROUND(CAST(cancelados AS FLOAT) / total * 100, 2) AS taxa_cancelamento
FROM contagem
ORDER BY taxa_cancelamento DESC
LIMIT 10
"""

df_resultado = pd.read_sql(query, conn)
df_resultado


,cancelados,total,product_category_name,taxa_cancelamento
0,1,9,pc_gamer,11.11
1,1,15,portateis_cozinha_e_preparadores_de_alimentos,6.67
2,2,64,dvds_blu_ray,3.13
3,5,194,construcao_ferramentas_seguranca,2.58
4,1,39,fraldas_higiene,2.56
5,4,238,construcao_ferramentas_jardim,1.68
6,11,680,instrumentos_musicais,1.62
7,7,553,livros_interesse_geral,1.27
8,3,238,eletrodomesticos_2,1.26
9,8,679,eletroportateis,1.18


In [68]:
query = """
SELECT c.customer_state, SUM(oi.price) AS faturamento,
RANK() OVER (ORDER BY SUM(oi.price) DESC ) AS posicao
FROM order_items AS oi
JOIN orders AS o
ON oi.order_id = o.order_id
JOIN customers AS c 
ON o.customer_id = c.customer_id
GROUP BY c.customer_state
""" 

df_resultado = pd.read_sql(query, conn)
df_resultado

,customer_state,faturamento,posicao
0,SP,5202955.05,1
1,RJ,1824092.67,2
2,MG,1585308.03,3
3,RS,750304.02,4
4,PR,683083.76,5
5,SC,520553.34,6
6,BA,511349.99,7
7,DF,302603.94,8
8,GO,294591.95,9
9,ES,275037.31,10


In [88]:
query = """
SELECT c.customer_state, 
       ROUND(AVG(JULIANDAY(o.order_delivered_customer_date) - JULIANDAY(o.order_purchase_timestamp)), 1) AS tempo_medio_dias
FROM orders AS o
JOIN customers AS c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
AND o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY tempo_medio_dias DESC
LIMIT 10
"""

df_resultado = pd.read_sql(query, conn)
df_resultado

,customer_state,tempo_medio_dias
0,RR,29.4
1,AP,27.2
2,AM,26.4
3,AL,24.5
4,PA,23.8
5,MA,21.6
6,SE,21.5
7,CE,21.3
8,AC,21.0
9,PB,20.4


In [94]:
query = """

SELECT s.seller_id, SUM(oi.price) AS faturamento_vendedor
FROM sellers AS s
JOIN order_items AS oi
ON s.seller_id = oi.seller_id
GROUP BY s.seller_id
ORDER BY faturamento_vendedor DESC
LIMIT 10
"""

df_resultado = pd.read_sql(query, conn)
df_resultado

,seller_id,faturamento_vendedor
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63
1,53243585a1d6dc2643021fd1853d8905,222776.05
2,4a3ca9315b744ce9f8e9374361493884,200472.92
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03
4,7c67e1448b00f6e969d365cea6b010ab,187923.89
5,7e93a43ef30c4f03f38b393420bc753a,176431.87
6,da8622b14eb17ae2831f4ac5b9dab84a,160236.57
7,7a67c85e85bb2ce8582c35f2203ad736,141745.53
8,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55
9,955fee9216a65b617aa5c0531780ce60,135171.70


In [118]:
query = """

WITH soma_aval_estado AS (

SELECT AVG(review_score) AS media_avaliacoes, s.seller_state, COUNT(orv.review_id) AS total_avaliacoes
FROM order_reviews AS orv
JOIN order_items AS oi
ON orv.order_id = oi.order_id


JOIN sellers AS s
ON oi.seller_id = s.seller_id

WHERE review_score IS NOT NULL

GROUP BY s.seller_state
)

SELECT * FROM soma_aval_estado
ORDER BY media_avaliacoes DESC

"""


df_resultado = pd.read_sql(query, conn)
df_resultado

,media_avaliacoes,seller_state,total_avaliacoes
0,4.500000,PA,8
1,4.469388,MS,49
2,4.267857,RN,56
3,4.254826,GO,518
4,4.214351,RS,2188
5,4.165517,MT,145
6,4.138298,CE,94
7,4.132584,PE,445
8,4.105868,MG,8794
9,4.101670,RJ,4790


In [132]:
query = """

WITH contagem_pedidos_id AS (

SELECT COUNT(o.order_id) AS pedidos, c.customer_unique_id
FROM orders AS o
JOIN customers AS c
ON o.customer_id = c.customer_id
GROUP BY c.customer_unique_id

)

SELECT COUNT(*) AS qtdePessoasRecorrentes FROM contagem_pedidos_id
WHERE pedidos >= 2


"""

df_resultado = pd.read_sql(query, conn)
df_resultado

,qtdePessoasRecorrentes
0,2997


In [134]:
query = """

SELECT STRFTIME('%Y-%m', order_purchase_timestamp) AS mes, SUM(oi.price) AS TOTAL
FROM orders AS o
JOIN order_items AS oi
on o.order_id = oi.order_id
GROUP BY mes
"""

df_resultado = pd.read_sql(query, conn)
df_resultado

,mes,TOTAL
0,2016-09,267.36
1,2016-10,49507.66
2,2016-12,10.90
3,2017-01,120312.87
4,2017-02,247303.02
5,2017-03,374344.30
6,2017-04,359927.23
7,2017-05,506071.14
8,2017-06,433038.60
9,2017-07,498031.48
